<a href="https://colab.research.google.com/github/drQedwards/ARC-AGI-Community-Leaderboard/blob/main/arc_agi2_enhanced_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time

TOTAL_SECONDS   = 12 * 3600
BUFFER_SECONDS  = 600
global_end_time = time.time() + TOTAL_SECONDS - BUFFER_SECONDS

print('Session ends at:', time.strftime('%H:%M:%S', time.localtime(global_end_time)))

In [ ]:
!pip uninstall -y tensorflow -q

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def grid_to_str(grid):
    return '\n'.join(''.join(str(int(c)) for c in row) for row in grid)

convert_grid_to_string = grid_to_str


def is_valid_grid(guess):
    return (
        isinstance(guess, np.ndarray)
        and guess.ndim == 2
        and all(1 <= x <= 30 for x in guess.shape)
    )

is_valid_solution = is_valid_grid


def shuffled(lst):
    return np.random.permutation(lst).tolist()


def permute_mod(a, descriptor, invert=False):
    perm = [int(ch) for ch in descriptor if ch.isdigit()]
    assert sorted(perm) == list(range(10))
    a = np.asarray(a)
    if a.ndim == 3:
        if not invert:
            perm = np.argsort(perm)
        a = a[..., perm]
    else:
        assert a.ndim == 2
        if invert:
            perm = np.argsort(perm)
        a = np.asarray(perm)[a]
    return a


def permute_rnd_all_(query):
    perm = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, perm))


class QwenFormatter:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query):
        return '<|im_start|>user\n' + grid_to_str(query[0]['input']) + '<|im_end|><|im_start|>assistant\n'

    def fmt_reply(self, reply):
        return grid_to_str(reply[0]) + '<|im_end|>'

    def fmt_train(self, train, last_is_challenge=False):
        if last_is_challenge:
            test  = train[-1]
            train = train[:-1]
        else:
            test = None
        parts = []
        for ex in train:
            parts.append(
                '<|im_start|>user\n'
                + grid_to_str(ex['input'])
                + '<|im_end|><|im_start|>assistant\n'
                + grid_to_str(ex['output'])
                + '<|im_end|>'
            )
        text = ''.join(parts)
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test['output']])
        return text

    def max_new_tokens(self):
        dummy = np.zeros([30, 30], dtype=int)
        return len(self.tokenizer.encode(self.fmt_reply([dummy]))) + 1

    def tokens_to_grid(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split('\n')
            rows  = [[int(ch) for ch in line if ch.isdigit()] for line in lines]
            rows  = [r for r in rows if r][:limit_rows]
            arr   = np.array(rows, dtype=int)
            if is_valid_grid(arr):
                return arr
        except Exception:
            pass
        return None

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        return self.tokens_to_grid(tokens, limit_rows)


class ArcDataset:
    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None:
            return a
        for op in key.split('.')[1:]:
            if   op == 'rot90':            a = np.rot90(a)
            elif op == 'transpose':        a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith(('copy', 'out', 'ex', 'run')): pass
            else: raise NotImplementedError('Unknown forward op: ' + repr(op))
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None:
            return a
        for op in key.split('.')[1:][::-1]:
            if   op == 'rot90':            a = np.rot90(a, k=3)
            elif op == 'transpose':        a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith(('copy', 'out', 'ex', 'run')): pass
            else: raise NotImplementedError('Unknown invert op: ' + repr(op))
        return a

    def __init__(self, queries, replies=None, keys=None, is_orig=False):
        if replies is None: replies = {}
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries  = queries if keys is None else {k: queries[k] for k in keys}
        self.replies  = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig  = is_orig
        self.keys     = sorted(queries.keys()) if keys is None else list(keys)
        self._transposed = None

    def change_keys(self, keys, keep_flags=False):
        flags = {'is_orig': self.is_orig} if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, path, keys=None):
        with open(path) as fh:
            raw = fh.read()
        return cls(queries=json.loads(raw), is_orig=True, keys=keys)

    def load_replies(self, path):
        print('*** Loading solutions from', repr(path), '...')
        with open(path) as fh:
            parsed = json.loads(fh.read())
        self.replies = {k: parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        idx = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys    = [k + '_' + str(i) for k, i in idx],
            queries = {k + '_' + str(i): {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in idx},
            replies = {k + '_' + str(i): [self.replies[k][i]] for k, i in idx if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries = {k: v for d in datasets for k, v in d.queries.items()},
            replies = {k: v for d in datasets for k, v in d.replies.items()},
            keys    = [k for d in datasets for k in d.keys],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries, replies, keys = {}, {}, []
        for k0 in self.keys:
            if descriptor is None:
                desc = ('copy{i}' if mod_func is np.copy else mod_func.__name__).format(i=i)
            elif isinstance(descriptor, str):
                desc = descriptor.format(i=i)
            else:
                desc = descriptor(self.queries[k0]).format(i=i)
            def apply(a, d=desc):
                return np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            prefix = 'I' if inputs_only else ''
            k1 = k0 if keep_key else k0 + '.' + prefix + desc
            keys.append(k1)
            queries[k1] = {
                split: [{t: (apply(v) if (t == 'input' or not inputs_only) else v) for t, v in pair.items()} for pair in pairs]
                for split, pairs in self.queries[k0].items()
            }
            if k0 in self.replies:
                replies[k1] = [apply(a) for a in self.replies[k0]]
        return self.__class__(queries=queries, replies=replies, keys=keys)

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False,
            keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        stack = mod_func.__name__.startswith('rot') if stack is None else stack
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys, new_queries, new_replies = [], {}, {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            sep = '-' if p.max() > 9 else ''
            nk  = key + '.ex' + sep.join(map(str, p.tolist()))
            new_keys.append(nk)
            new_queries[nk] = {
                split: (np.array(v, dtype=object)[p].tolist() if split == 'train' else v)
                for split, v in self.queries[key].items()
            }
            if key in self.replies:
                new_replies[nk] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_length(self, key, formatter, name, max_of_transposed=False):
        if formatter is None:
            if name == 'input':
                return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name == 'reply':
                return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else:
                raise ValueError(name)
        datasets = [self]
        if max_of_transposed:
            if self._transposed is None:
                self._transposed = self.mod(np.transpose, keep=False, keep_key=True)
            datasets.append(self._transposed)
        return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        tmp = self.change_keys(self.keys)
        new_keys, new_queries, new_replies = [], {}, {}
        for key in self.keys:
            reply = tmp.replies.get(key)
            while max_len < tmp.get_length(key, formatter=formatter, name=name):
                query = tmp.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = key + '.ex' + ''.join(map(str, range(len(query['train']))))
                parts = key.split('.')
                assert parts[-1].startswith('ex')
                trimmed = parts[-1][2:-1] if from_end else parts[-1][3:]
                key = '.'.join(parts[:-1] + ['ex' + trimmed])
                tmp.queries[key] = {
                    split: (v[:-1] if from_end else v[1:]) if split == 'train' else v
                    for split, v in query.items()
                }
                if reply is not None:
                    tmp.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = tmp.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def get(self, key, formatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text  = (train + query + reply) if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train + query, text=text)

    def as_list(self, formatter):
        return [self.get(k, formatter) for k in self.keys]

    def get_submission(self, results=None):
        assert self.is_orig
        sub = {k: [{'attempt_1': [[0]], 'attempt_2': [[0]]} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None:
            self.fill_submission(results, sub)
        return sub

    @staticmethod
    def fill_submission(results, submission):
        print('*** Writing', len(results), 'outputs into submission ...')
        for k, v in results.items():
            base_id, base_nr = k.rsplit('_', 1)
            target = submission[base_id][int(base_nr)]
            for i, grid in enumerate(v[:len(target)]):
                target['attempt_' + str(i + 1)] = grid.tolist()

    def validate_submission(self, submission):
        assert self.is_orig
        score = 0.0
        for k, outputs in self.replies.items():
            for idx, ref in enumerate(outputs):
                for attempt in ('attempt_1', 'attempt_2'):
                    if np.array_equal(ref, submission[k][idx][attempt]):
                        score += 1.0 / len(outputs)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
# arc_decoder.py  [DIFF 5: weighted ensemble scoring]
import os, bz2, pickle
import numpy as np


def to_hashable(grid):
    return tuple(map(tuple, grid))


def rank_by_score(guesses, score_fn):
    buckets = {}
    for g in guesses.values():
        h = to_hashable(g['solution'])
        bucket = buckets.setdefault(h, ([], g['solution']))
        bucket[0].append(g)
    ranked = sorted(
        ((score_fn(members), sol) for members, sol in buckets.values()),
        key=lambda x: x[0], reverse=True,
    )
    return [sol for _, sol in ranked]


def _score_probmul(guesses, baseline=3.0):
    beam_part = float(np.sum([baseline - g['beam_score'] for g in guesses]))
    aug_part  = float(np.mean([np.sum([baseline - s for s in g['score_aug']]) for g in guesses]))
    return beam_part + aug_part


def _score_vote(guesses):
    vote_count = len(guesses)
    aug_mean   = float(np.mean([np.mean(g['score_aug']) for g in guesses]))
    return vote_count - aug_mean


def score_probmul(guesses):
    return rank_by_score(guesses, _score_probmul)

score_full_probmul_3 = score_probmul


def score_vote(guesses):
    return rank_by_score(guesses, _score_vote)

score_kgmon = score_vote


def score_ensemble(guesses):
    # [DIFF 5] Weighted blend: 0.7 * probmul + 0.3 * vote
    # Replaces naive interleave which could surface a mediocre candidate
    # from one ranker ahead of a strong one from the other.
    def _combined(members):
        return 0.7 * _score_probmul(members) + 0.3 * _score_vote(members)
    return rank_by_score(guesses, _combined)


SELECTION_ALGORITHMS = [score_probmul, score_vote, score_ensemble]


class ArcDecoder:
    def __init__(self, dataset, n_guesses=2):
        self.dataset         = dataset
        self.n_guesses       = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_tag=''):
        for fname in os.listdir(store):
            path     = os.path.join(store, fname)
            base_key = fname.split('.')[0]
            self.decoded_results.setdefault(base_key, {})
            with bz2.BZ2File(path) as fh:
                outputs = pickle.load(fh)
            for idx, sample in enumerate(outputs):
                self.decoded_results[base_key][fname + run_tag + '.out' + str(idx)] = sample

    def run_selection(self, algorithm=None):
        if algorithm is None:
            algorithm = score_vote
        return {bk: algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_selection_algo(self, selection_algorithm=None):
        return self.run_selection(selection_algorithm)

    def benchmark_selection_algos(self):
        print('=' * 60)
        print('  Selection-algorithm benchmark')
        print('=' * 60)
        labels, tasks_per_puzzle = {}, {}
        n_correct_keys, n_total_keys, correct_beam_scores = 0, 0, []

        for base_key, base_vals in self.decoded_results.items():
            puzzle_id, sub_idx = base_key.split('_')
            tasks_per_puzzle[puzzle_id] = max(tasks_per_puzzle.get(puzzle_id, 0), int(sub_idx) + 1)
            ref = self.dataset.replies[base_key][0]
            labels[base_key] = ref

            for sub_key, sample in base_vals.items():
                sol    = sample['solution']
                bscore = sample['beam_score']
                amean  = float(np.mean(sample['score_aug']))
                if np.shape(ref) != np.shape(sol):
                    tag = 'bad_shape'
                elif np.array_equal(ref, sol):
                    tag = 'CORRECT'
                    n_correct_keys += 1
                    correct_beam_scores.append(bscore)
                else:
                    tag = 'wrong'
                if tag == 'CORRECT':
                    sz = str(sol.shape[0]) + 'x' + str(sol.shape[1])
                    print('  ' + tag + ' beam=' + str(round(bscore, 4)) + ' aug=' + str(round(amean, 4)) + ' ' + sz + ' [' + sub_key + ']')
                n_total_keys += 1

        print('Sub-key recall :', n_correct_keys, '/', n_total_keys)
        if correct_beam_scores:
            print('Beam score (correct) mean=' + str(round(float(np.mean(correct_beam_scores)), 4)) + '  max=' + str(round(float(np.max(correct_beam_scores)), 4)))

        n_puzzles = len(tasks_per_puzzle)
        print()
        for algo in SELECTION_ALGORITHMS:
            name     = algo.__name__
            selected = self.run_selection(algo)
            solved   = {k for k, cands in selected.items() if any(np.array_equal(c, labels[k]) for c in cands[:self.n_guesses])}
            score    = sum(1.0 / tasks_per_puzzle[k.split('_')[0]] for k in solved)
            print('  [' + name + ']  ' + str(round(score, 1)) + ' / ' + str(n_puzzles) + '  (' + str(round(100 * score / max(n_puzzles, 1), 1)) + '%)')

In [ ]:
%%writefile arc_solver.py
# arc_solver.py
# [DIFF 1] lora_alpha: 32 -> 256
# [DIFF 2] eval augmentation: n=2 -> n=6
# [DIFF 3] beam threshold: -log(0.2) -> -log(0.1)
# [DIFF 4] dynamic per-puzzle time budget
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc, os, io, time, torch, numpy as np, logging, bz2, pickle
from datasets import Dataset
from collections import defaultdict
from typing import Any, Union
from transformers import DataCollatorForLanguageModeling
from contextlib import redirect_stdout, redirect_stderr
from peft import get_peft_model_state_dict, set_peft_model_state_dict

logging.disable(logging.WARNING)

ARC_VOCAB = {str(d): d for d in range(10)}
ARC_VOCAB.update({'\u010a': 10, '<|im_end|>': 15})
ARC_TOKENS         = list(ARC_VOCAB.values())
USER_TOKEN_ID      = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID             = 13
EOS_ID             = 15


class FixedUnslothTrainer(UnslothTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels') if (self.label_smoother and 'labels' in inputs) else None
        outputs = model(**inputs)
        if labels is not None:
            raw   = self.accelerator.unwrap_model(model)
            shift = hasattr(raw, '_get_name') and 'unsloth' in raw._get_name().lower()
            loss  = self.label_smoother(outputs, labels, shift_labels=shift) if shift else self.label_smoother(outputs, labels)
        else:
            loss = outputs['loss'] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, 'clone'):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class ArcDataCollator(DataCollatorForLanguageModeling):
    def torch_call(self, examples):
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            ids    = batch['input_ids'][i]
            labels = ids.clone()
            u_pos  = np.where(ids.numpy() == USER_TOKEN_ID)[0].tolist()
            a_pos  = np.where(ids.numpy() == ASSISTANT_TOKEN_ID)[0].tolist()
            starts = sorted(u_pos + a_pos)
            ends   = np.where(ids.numpy() == EOS_ID)[0]
            batch['labels'][i, :] = -100
            for turn_idx, (s, e) in enumerate(zip(starts, ends)):
                assert s < e
                if turn_idx % 2 == 1:
                    s += 2
                    e += 1
                    batch['labels'][i, s:e] = labels[s:e]
        return batch


def _dfs_step(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n   = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes   = defaultdict(list)
    candidates = {i: [] for i in range(n)}
    for i in range(n):
        for t in ARC_TOKENS:
            sc = nll[i, t].item()
            if sc >= max_score: continue
            if t == EOS_ID:
                suffixes[i].append((sc, [t]))
            elif max_new_tokens > 1:
                candidates[i].append((sc, t))
        candidates[i].sort(key=lambda x: x[0])

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_toks, batch_scores, n_alive = [], [], 0
        for i in range(n):
            if candidates[i]:
                sc, t = candidates[i].pop(0)
                batch_toks.append(t)
                batch_scores.append(sc)
                n_alive += 1
            else:
                batch_toks.append(PAD_ID)
                batch_scores.append(1e6)
        if n_alive == 0: break
        out = model(
            input_ids       = torch.tensor(batch_toks, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids    = torch.full((n, 1), pos, device=model.device),
            past_key_values = cache,
            return_dict=True, use_cache=True,
        )
        child_suffixes = _dfs_step(model, out.logits[:, -1], max_new_tokens - 1, max_score,
                                   batch_scores, pos + 1, out.past_key_values, start_time, end_time)
        for bid, beams in child_suffixes.items():
            for sc, toks in beams:
                toks.insert(0, batch_toks[bid])
                suffixes[bid].append((sc, toks))
    return suffixes


@torch.no_grad()
def beam_search(model, prefix_token_lists, max_new_tokens, max_score, end_time):
    # Pad all prefixes to the same length (ragged sequences crash torch.tensor)
    max_len = max(len(t) for t in prefix_token_lists)
    padded  = [t + [PAD_ID] * (max_len - len(t)) for t in prefix_token_lists]
    input_ids = torch.tensor(padded, device=model.device, dtype=torch.long)
    out       = model(input_ids=input_ids, return_dict=True, use_cache=True)
    raw = _dfs_step(model, out.logits[:, -1], max_new_tokens, max_score,
                    [0.0] * input_ids.size(0), input_ids.size(1),
                    out.past_key_values, time.time(), end_time)
    return [(bid, sorted(beams, key=lambda x: x[0])) for bid, beams in raw.items()]

inference_turbo_dfs = beam_search


@torch.no_grad()
def score_candidates(queries, answers, tokenizer, model):
    q_toks, a_toks, all_toks, lengths = [], [], [], []
    for q, a in zip(queries, answers):
        qt = tokenizer.encode(q)
        at = tokenizer.encode(a)
        q_toks.append(qt); a_toks.append(at)
        all_toks.append(qt + at); lengths.append(len(qt) + len(at))
    max_len   = max(lengths)
    padded    = [t + [PAD_ID] * (max_len - len(t)) for t in all_toks]
    input_ids = torch.tensor(padded, device=model.device, dtype=torch.long)
    out       = model(input_ids=input_ids, return_dict=True, use_cache=True)
    log_probs = out.logits.float().cpu().log_softmax(-1)
    results = []
    for lp, qt, at in zip(log_probs, q_toks, a_toks):
        ql     = len(qt)
        ans_lp = lp[ql - 1: ql - 1 + len(at)]
        nll    = -ans_lp[torch.arange(len(at)), at].sum().item()
        results.append(nll)
    return results

calc_scores = score_candidates


def worker(rank, queue, end_time):
    is_rerun = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

    # [DIFF 1] lora_alpha 32 -> 256: fixes weak alpha/rank ratio (0.125 -> 1.0)
    peft_cfg = dict(
        r                          = 256,
        target_modules             = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                      'gate_proj', 'up_proj', 'down_proj',
                                      'embed_tokens', 'lm_head'],
        lora_alpha                 = 256,
        lora_dropout               = 0.0,
        bias                       = 'none',
        use_gradient_checkpointing = False,
        random_state               = 42,
        use_rslora                 = True,
        loftq_config               = None,
    )

    train_cfg = dict(
        per_device_train_batch_size  = 1,
        per_device_eval_batch_size   = 1,
        gradient_accumulation_steps  = 1,
        num_train_epochs             = 1,
        warmup_steps                 = 0,
        warmup_ratio                 = 0.1,
        max_grad_norm                = 1.0,
        learning_rate                = 5e-5,
        optim                        = 'adamw_torch',
        weight_decay                 = 0.0,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
        report_to                    = 'none',
        save_strategy                = 'no',
        eval_strategy                = 'no',
        logging_strategy             = 'no',
        fp16                         = False,
        bf16                         = True,
        fsdp                         = '',
        ddp_find_unused_parameters   = False,
        dataloader_num_workers       = 0,
        gradient_checkpointing       = False,
    )

    MAX_SEQ = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name                 = '/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1',
        full_finetuning            = False,
        load_in_4bit               = False,
        local_files_only           = True,
        use_gradient_checkpointing = False,
        max_seq_length             = MAX_SEQ,
    )
    model = FastLanguageModel.get_peft_model(model, **peft_cfg)

    for _, p in model.named_parameters():
        if p.dtype == torch.float32:
            p.data = p.data.to(torch.bfloat16)

    init_weights = {k: v.clone().detach() for k, v in get_peft_model_state_dict(model, adapter_name='default').items()}
    collator  = ArcDataCollator(tokenizer=tokenizer, mlm=False)
    formatter = QwenFormatter(tokenizer=tokenizer)
    MAX_NEW   = formatter.max_new_tokens()

    # [DIFF 3] Widen beam threshold from p=0.2 to p=0.1
    MAX_SCORE = -np.log(0.1)

    test_path = (
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json'
        if is_rerun else
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json'
    )
    arc_test = ArcDataset.from_file(test_path)
    out_dir  = '/kaggle/inference_outputs'
    os.makedirs(out_dir, exist_ok=True)

    while not queue.empty():
        if time.time() > end_time:
            print('[GPU ' + str(rank) + '] time limit reached - stopping.')
            break

        key = queue.get()
        if key is None: break

        t0 = time.time()
        torch.cuda.reset_peak_memory_stats()
        print('[GPU ' + str(rank) + '] starting puzzle ' + key)

        set_peft_model_state_dict(model, init_weights.copy(), adapter_name='default')
        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test.change_keys([key])
        train_ds  = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds  = train_ds.cut_to_len(formatter=formatter, name='text', max_len=MAX_SEQ)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = FixedUnslothTrainer(
                model=model, tokenizer=tokenizer, data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field='text', max_seq_length=MAX_SEQ,
                args=UnslothTrainingArguments(**train_cfg),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)
        gc.collect()
        torch.cuda.empty_cache()
        mem_train = torch.cuda.max_memory_allocated() // 1024 ** 2
        print('[GPU ' + str(rank) + '] training peak memory: ' + str(mem_train) + ' MB  stats: ' + str(stats))
        torch.cuda.reset_peak_memory_stats()

        puzzle_multi = puzzle_ds.split_multi_replies()

        # [DIFF 2] n=2 -> n=6: 16 eval views -> 48
        eval_ds = puzzle_multi.augment(n=6, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name='input', max_len=MAX_SEQ - MAX_NEW)

        by_test = defaultdict(list)
        for sk in sorted(eval_ds.keys):
            tid = sk.split('.')[0].split('_')[1]
            by_test[tid].append(sk)

        batches = []
        for sks in by_test.values():
            for offsets in ([0, 4], [2, 6]):
                batch = []
                for off in offsets: batch.extend(sks[off: off + 2])
                batches.append(batch)
            for offsets in ([8, 12], [10, 14]):
                batch = []
                for off in offsets: batch.extend(sks[off: off + 2])
                batches.append(batch)

        with torch.inference_mode():
            scored_cache = {}

            for batch_sks in batches:
                elapsed = time.time() - t0

                # [DIFF 4] Dynamic budget: min(1200s, time until global end - 2min)
                puzzle_budget = min(1200.0, (end_time - t0) - 120.0)
                if elapsed > puzzle_budget or time.time() > end_time:
                    print('[GPU ' + str(rank) + '] puzzle ' + key + ' timed out after ' + str(round(elapsed)) + 's (budget=' + str(round(puzzle_budget)) + 's)')
                    break

                print('[GPU ' + str(rank) + '] decoding ' + str(batch_sks))
                prefix_toks = [tokenizer.encode(eval_ds.get(sk, formatter)['input']) for sk in batch_sks]
                dfs_out     = beam_search(model, prefix_toks, MAX_NEW, MAX_SCORE, end_time)

                for bid, beams in dfs_out:
                    sk   = batch_sks[bid]
                    bk   = sk.split('.')[0]
                    hits = []

                    for beam_score, toks in beams:
                        arr = formatter.tokens_to_grid(toks)
                        if arr is None: continue
                        sol = puzzle_multi.invert_mod(arr, sk, inv_perm=True)
                        gid = (bk, tuple(map(tuple, sol)))

                        if gid in scored_cache:
                            aug_scores = scored_cache[gid]
                        else:
                            print('[GPU ' + str(rank) + '] verifying ' + sk + ' candidate #' + str(len(hits)))
                            aug_ds = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_multi.queries[bk]},
                                replies={bk: [sol.tolist()]},
                            )
                            aug_ds = aug_ds.augment(seed=hash(bk) % (1024 ** 2))
                            aug_ds = aug_ds.cut_to_len(formatter=formatter, name='input', max_len=MAX_SEQ - MAX_NEW)
                            qs, ans = zip(*[(s['input'], s['reply']) for s in aug_ds.as_list(formatter)])
                            aug_scores = (
                                score_candidates(list(qs[:4]),  list(ans[:4]),  tokenizer, model)
                                + score_candidates(list(qs[4:]), list(ans[4:]), tokenizer, model)
                            )
                            scored_cache[gid] = aug_scores

                        hits.append({'beam_score': beam_score, 'score_aug': aug_scores, 'solution': sol})

                    if hits:
                        out_path = os.path.join(out_dir, sk)
                        with bz2.BZ2File(out_path, 'w') as fh:
                            pickle.dump(hits, fh)

        mem_infer = torch.cuda.max_memory_allocated() // 1024 ** 2
        elapsed   = time.time() - t0
        print('[GPU ' + str(rank) + '] puzzle ' + key + ' done - ' + str(round(elapsed)) + 's  infer peak ' + str(mem_infer) + ' MB')

In [ ]:
%%writefile starter.py
import os, time, json, torch, argparse
import torch.multiprocessing as mp


def gpu_worker(rank, queue, end_time):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(rank)
    torch.set_default_device('cpu')
    if rank > 0:
        sentinel = '/kaggle/worker' + str(rank - 1) + '.ready'
        while not os.path.exists(sentinel):
            time.sleep(5)
    from arc_solver import worker
    with open('/kaggle/worker' + str(rank) + '.ready', 'w') as fh:
        fh.write('ok')
    print('[GPU ' + str(rank) + '] initialised - starting puzzle loop')
    worker(rank, queue, end_time)
    print('[GPU ' + str(rank) + '] finished')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--end-time', type=float, default=0.0)
    parser.add_argument('--gpus',     type=int,   default=4)
    args = parser.parse_args()

    is_rerun  = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
    test_path = (
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json'
        if is_rerun else
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json'
    )
    with open(test_path) as fh:
        tasks = json.load(fh)

    manager = mp.Manager()
    queue   = manager.Queue()
    for key in sorted(tasks.keys()):
        if not is_rerun:
            if key not in {'0934a4d8', '36a08778', '981571dc', 'aa4ec2a5'}:
                continue
        queue.put(key)
    for _ in range(args.gpus):
        queue.put(None)

    mp.spawn(gpu_worker, args=(queue, args.end_time), nprocs=args.gpus)

In [ ]:
import subprocess, sys, os

cmd = [
    sys.executable, 'starter.py',
    '--end-time', str(global_end_time),
    '--gpus', '4',
]
env = {
    **os.environ,
    'UNSLOTH_DISABLE_STATISTICS': '1',
    'TRITON_PTXAS_PATH': '/usr/local/cuda/bin/ptxas',
    'OMP_NUM_THREADS': '12',
}
subprocess.run(cmd, env=env, check=True)

In [ ]:
import os, json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_ensemble

is_rerun = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

if is_rerun:
    data = ArcDataset.from_file(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json'
    )
else:
    data = ArcDataset.from_file(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json'
    )
    data.load_replies(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json'
    )

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results('/kaggle/inference_outputs')

# [DIFF 5] weighted ensemble (0.7 probmul + 0.3 vote)
submission = data.get_submission(decoder.run_selection(score_ensemble))

with open('submission.json', 'w') as fh:
    json.dump(submission, fh)

print('submission.json written.')

if not is_rerun:
    decoder.benchmark_selection_algos()
    with open('submission.json') as fh:
        reloaded = json.load(fh)
    score = data.validate_submission(reloaded)
    print('Final validation score:', round(score, 4))